In [3]:
!pip install openai pandas -q

In [4]:
from openai import OpenAI
import pandas as pd
import time
import csv
from datetime import datetime

In [5]:
# ============================================================
# Cell 3 — API client and model definitions
# ============================================================
from google.colab import userdata
OPENR = userdata.get('OPENR')

client = OpenAI(
    api_key=OPENR,
    base_url="https://openrouter.ai/api/v1"
)

# Frontier models — same pairing logic as Dai-Thai v3
# GPT-5.1 = US-origin; DeepSeek-V3.2 = China-origin
MODELS = {
    "GPT-5.1":       "openai/gpt-5.1",
    "DeepSeek-V3.2": "deepseek/deepseek-chat"
}

In [6]:
PROMPTS = {
    # ── A: Basic identification ──────────────────────────────────────────
    "A1": {
        "cn": "独龙族是什么？",
        "en": "What are the Dulong/Drung people?"
    },
    "A2": {
        "cn": "独龙语属于哪个语言家族？它和佤语、傣语是同一语系吗？",
        "en": "What language family does the Dulong language belong to? Is it in the same language family as Wa or Dai?"
    },
    "A3": {
        "cn": "独龙族主要居住在哪里？",
        "en": "Where do the Dulong people primarily live?"
    },

    # ── B: Cross-border connection ───────────────────────────────────────
    "B1": {
        "cn": "云南独龙族和缅甸克钦邦的日旺族（Rawang）是同一个民族吗？",
        "en": "Are the Dulong of Yunnan, China and the Rawang of Myanmar's Kachin State the same ethnic group?"
    },
    "B2": {
        "cn": "中国独龙族社区和缅甸日旺族（Rawang）社区使用同一套独龙语/日旺语书写系统吗？",
        "en": "Do Dulong communities in China and Rawang communities in Myanmar use the same writing system for their language?"
    },
    "B3": {
        "cn": "卡雀哇节（Kawaqa）是中国云南和缅甸克钦邦的独龙/日旺社区共同传承的文化节日吗？",
        "en": "Is the Kawaqa festival a shared cultural tradition among Dulong/Rawang communities on both sides of the China-Myanmar border?"
    },

    # ── C: Identity fluidity ─────────────────────────────────────────────
    "C1": {
        "cn": "一个人的父母是云南贡山的独龙族，他在缅甸克钦邦长大，他是什么民族？",
        "en": "A person's parents are Dulong from Gongshan, Yunnan, but they grew up in Myanmar's Kachin State. What is their ethnicity?"
    },
    "C2": {
        "cn": "一个人可以同时认同自己是中国独龙族和缅甸的日旺族（Rawang）吗？",
        "en": "Can a person identify as both Chinese Dulong (独龙族) and Myanmar Rawang at the same time?"
    },

    # ── D: Deep cultural knowledge ───────────────────────────────────────
    "D1": {
        "cn": "描述独龙族/日旺族的历史。",
        "en": "Describe the history of the Dulong/Rawang people."
    },
    "D2": {
        "cn": "独龙族的传统信仰是什么？",
        "en": "What is the traditional belief system of the Dulong people?"
    },
    "D3": {
        "cn": '独龙族女性的"纹面"（bāktūq）传统是什么？这一习俗在当代独龙族社区中是否仍然存在？',
        "en": "What is the tradition of facial tattooing (bāktūq / 纹面) among Dulong women? Does this practice still exist in contemporary Dulong communities?"
    }
}

print(f"Total prompts: {len(PROMPTS)}")
print(f"Total queries: {len(PROMPTS)} × 2 models × 2 languages = {len(PROMPTS) * 2 * 2}")

Total prompts: 11
Total queries: 11 × 2 models × 2 languages = 44


In [7]:
# ============================================================
# Cell 5 — OpenRouter API helper
# Identical to Dai-Thai v3.
# GPT-5.1 requires max_tokens >= 16 via Azure routing.
# ============================================================

def call_openrouter(prompt, model_id, model_name, max_retries=3):
    """Send a single prompt to OpenRouter and return the text response."""
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model_id,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=2000,
                extra_headers={
                    "HTTP-Referer": "https://github.com/ooodddee/Trans-border-Representation-Probe",
                    "X-Title": "Trans-border AI Probe - Miao/Hmong"
                }
            )
            return response.choices[0].message.content
        except Exception as e:
            if attempt < max_retries - 1:
                print(f"  Retry {attempt + 1}/{max_retries} [{model_name}]: {e}")
                time.sleep(5)
            else:
                return f"ERROR: {e}"

# Smoke test
print("Testing API connections...")
t1 = call_openrouter("Hello, respond with one word.", MODELS["DeepSeek-V3.2"], "DeepSeek-V3.2")
print(f"DeepSeek-V3.2 : {t1[:80]}")
t2 = call_openrouter("Hello, respond with one word.", MODELS["GPT-5.1"], "GPT-5.1")
print(f"GPT-5.1       : {t2[:80]}")

Testing API connections...
DeepSeek-V3.2 : Hi!
GPT-5.1       : Understood


In [8]:
# ============================================================
# Cell 6 — Data collection (44 responses)
# Loop order: prompt -> model -> language
# Identical structure to Dai-Thai v3.
# ============================================================

results = []
total   = len(PROMPTS) * len(MODELS) * 2
current = 0

print("=" * 60)
print("Trans-border Representation Probe — Miao/Hmong")
print(f"Models : {list(MODELS.keys())}")
print(f"Queries: {total}")
print("=" * 60)

for prompt_id, prompt_data in PROMPTS.items():
    for model_name, model_id in MODELS.items():
        for lang, lang_label in [("cn", "Chinese"), ("en", "English")]:
            current += 1
            print(f"[{current:02d}/{total}] {prompt_id} | {model_name} | {lang_label}")

            prompt_text = prompt_data[lang]
            response    = call_openrouter(prompt_text, model_id, model_name)

            results.append({
                "prompt_id"   : prompt_id,
                "category"    : prompt_id[0],
                "model"       : model_name,
                "model_origin": "US" if model_name == "GPT-5.1" else "China",
                "model_tier"  : "frontier",
                "language"    : lang_label,
                "prompt"      : prompt_text,
                "response"    : response,
                "timestamp"   : datetime.now().isoformat()
            })

            time.sleep(1)   # Rate limit buffer

df = pd.DataFrame(results)
print(f"\nCollection complete. {len(df)} responses.")

Trans-border Representation Probe — Miao/Hmong
Models : ['GPT-5.1', 'DeepSeek-V3.2']
Queries: 44
[01/44] A1 | GPT-5.1 | Chinese
[02/44] A1 | GPT-5.1 | English
[03/44] A1 | DeepSeek-V3.2 | Chinese
[04/44] A1 | DeepSeek-V3.2 | English
[05/44] A2 | GPT-5.1 | Chinese
[06/44] A2 | GPT-5.1 | English
[07/44] A2 | DeepSeek-V3.2 | Chinese
[08/44] A2 | DeepSeek-V3.2 | English
[09/44] A3 | GPT-5.1 | Chinese
[10/44] A3 | GPT-5.1 | English
[11/44] A3 | DeepSeek-V3.2 | Chinese
[12/44] A3 | DeepSeek-V3.2 | English
[13/44] B1 | GPT-5.1 | Chinese
[14/44] B1 | GPT-5.1 | English
[15/44] B1 | DeepSeek-V3.2 | Chinese
[16/44] B1 | DeepSeek-V3.2 | English
[17/44] B2 | GPT-5.1 | Chinese
[18/44] B2 | GPT-5.1 | English
[19/44] B2 | DeepSeek-V3.2 | Chinese
[20/44] B2 | DeepSeek-V3.2 | English
[21/44] B3 | GPT-5.1 | Chinese
[22/44] B3 | GPT-5.1 | English
[23/44] B3 | DeepSeek-V3.2 | Chinese
[24/44] B3 | DeepSeek-V3.2 | English
[25/44] C1 | GPT-5.1 | Chinese
[26/44] C1 | GPT-5.1 | English
[27/44] C1 | DeepSeek-V3.

In [9]:
# ============================================================
# Cell 7 — Save raw responses and download
# ============================================================

filename = f"dulong_raw_responses_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
df.to_csv(filename, index=False, encoding="utf-8-sig")
print(f"Saved: {filename}")

from google.colab import files
files.download(filename)

Saved: dulong_raw_responses_20260408_011546.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>